# Gold Layer РІР‚вЂќ All Feature Engineering

In [1]:
import duckdb
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import date

import textstat
from langdetect import detect, LangDetectException, DetectorFactory
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DetectorFactory.seed = 0

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
con = duckdb.connect('../../ProjectData.duckdb')
con.execute('CREATE SCHEMA IF NOT EXISTS gold')

# Load all three silver splits and concat into one DataFrame.
silver_parts = []
for split in ['train', 'test', 'validation']:
    p = Path(f'../../data/silver/{split}/cleaned_{split}.parquet')
    if not p.exists():
        raise FileNotFoundError(f'Silver split not found: {p}. Run clean_transform_to_silver.ipynb first.')
    df = pd.read_parquet(p)
    if 'label' not in df.columns:
        df['label'] = None
    silver_parts.append(df)

silver_df = pd.concat(silver_parts, ignore_index=True)

silver_df['_index'] = range(len(silver_df))

print(f'Rows: {len(silver_df):,}  |  Splits: {silver_df["dataset_split"].value_counts().to_dict()}')
print(f'Columns: {list(silver_df.columns)}')

con.register('silver_df_view', silver_df)
con.execute('CREATE OR REPLACE VIEW silver_data AS SELECT * FROM silver_df_view')

Rows: 11,999  |  Splits: {'train': 9613, 'validation': 1249, 'test': 1137}
Columns: ['Row_id', 'product_id', 'product_parent', 'product_title', 'vine', 'verified_purchase', 'review_headline', 'review_body', 'review_date', 'marketplace_id', 'product_category_id', 'label', 'dataset_split', '_source_file', '_line_number', 'record_origin', 'detected_issues', '_ingested_at', '_index']


## Temporal Features

In [3]:
con.execute('DROP TABLE IF EXISTS gold.review_temporal_features')
con.execute("""
    CREATE TABLE gold.review_temporal_features AS
    WITH base AS (
        SELECT *,
            MAX(review_date) OVER ()                              AS dataset_max_date,
            ROW_NUMBER() OVER (
                PARTITION BY product_parent ORDER BY review_date
            )                                                     AS review_relative_rank,
            COUNT(*) OVER (PARTITION BY product_parent)           AS product_review_count,
            MIN(review_date) OVER (PARTITION BY product_parent)   AS first_review_date,
            MAX(review_date) OVER (PARTITION BY product_parent)   AS last_review_date,
            COUNT(*) OVER (
                PARTITION BY product_parent, review_date
            )                                                     AS reviews_same_day
        FROM silver_data
    ),
    ranked AS (
        SELECT *,
            DATE_DIFF('day', review_date, dataset_max_date)       AS review_age_days,
            DATE_DIFF('day', first_review_date, review_date)      AS days_since_first_review,
            MONTH(review_date)                                    AS review_month,
            DAYOFWEEK(review_date)                                AS review_dayofweek,
            CASE
                WHEN review_relative_rank <= CEIL(product_review_count * 0.10)
                THEN TRUE ELSE FALSE
            END                                                   AS is_early_review,
            product_review_count * 1.0
                / NULLIF(DATE_DIFF('day', first_review_date, last_review_date), 0)
                AS reviews_per_day
        FROM base
    )
    SELECT * FROM ranked
""")

con.execute("""
    SELECT
        COUNT(*)                                           AS total_rows,
        SUM(CASE WHEN is_early_review THEN 1 END)         AS early_reviews,
        ROUND(AVG(review_age_days), 1)                    AS avg_age_days,
        ROUND(AVG(days_since_first_review), 1)            AS avg_days_since_first,
        MAX(product_review_count)                         AS max_reviews_per_product,
        ROUND(AVG(reviews_per_day), 3)                    AS avg_reviews_per_day
    FROM gold.review_temporal_features
""").df()

,total_rows,early_reviews,avg_age_days,avg_days_since_first,max_reviews_per_product,avg_reviews_per_day
0,11999,5735.0,1165.1,462.9,53,0.025


## Lexical & Language Features

In [4]:
def detect_language(text):
    if text is None or str(text).strip() == '' or len(str(text).strip()) < 10:
        return 'unknown'
    try:
        return detect(str(text))
    except LangDetectException:
        return 'unknown'

def type_token_ratio(text):
    if text is None or str(text).strip() == '':
        return None
    tokens = str(text).lower().split()
    return len(set(tokens)) / len(tokens) if tokens else None

silver_df['detected_language'] = silver_df['review_body'].apply(detect_language)
print('\nLanguage distribution (top 10):')
print(silver_df['detected_language'].value_counts().head(10))

silver_df['type_token_ratio'] = silver_df['review_body'].apply(type_token_ratio)


Language distribution (top 10):
detected_language
en         3991
fr         3849
de         2849
hu          868
unknown     220
vi           80
ca           19
it           18
af           18
da           15
Name: count, dtype: int64


In [5]:
con.register('silver_with_lang', silver_df)

con.execute('DROP TABLE IF EXISTS gold.review_lexical_features')
con.execute("""
    CREATE TABLE gold.review_lexical_features AS
    WITH text_stats AS (
        SELECT
            _index, product_id, product_parent, product_title,
            vine, verified_purchase, review_headline, review_body,
            review_date, marketplace_id, product_category_id,
            label, record_origin, detected_language, type_token_ratio,

            ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' '))     AS body_word_count,
            CASE
                WHEN review_headline IS NOT NULL AND TRIM(review_headline) != ''
                THEN ARRAY_LENGTH(STRING_SPLIT(TRIM(review_headline), ' '))
                ELSE 0
            END                                                     AS headline_word_count,
            (LENGTH(review_body) - LENGTH(REPLACE(review_body, '!', ''))) AS exclamation_count,
            (LENGTH(review_body) - LENGTH(REPLACE(review_body, '?', ''))) AS question_count,
            GREATEST(
                LENGTH(review_body) - LENGTH(REPLACE(review_body, '.', '')), 1
            )                                                        AS sentence_count_approx,
            (LENGTH(REPLACE(review_body, ' ', '')) * 1.0)
                / NULLIF(ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' ')), 0)
                                                                     AS avg_word_length,
            CONCAT(vine, '_', CAST(marketplace_id AS VARCHAR))       AS vine_x_marketplace,
            CONCAT(verified_purchase, '_', CAST(product_category_id AS VARCHAR))
                                                                     AS verified_x_category,
            COALESCE(ARRAY_LENGTH(regexp_extract_all(review_body, '&[a-zA-Z]+;|&#[0-9]+')), 0) * 1.0
                / NULLIF(ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' ')), 0)
                                                                     AS html_entity_density,
            COALESCE(ARRAY_LENGTH(regexp_extract_all(review_body, '<br\s*/?>')), 0)
                                                                     AS paragraph_break_count,
            CASE
                WHEN COALESCE(ARRAY_LENGTH(regexp_extract_all(review_body, '<br\s*/?>')), 0) >= 2
                THEN TRUE ELSE FALSE
            END                                                      AS has_structured_body
        FROM silver_with_lang
        WHERE review_body IS NOT NULL AND TRIM(review_body) != ''
    ),
    lang_norms AS (
        SELECT
            *,
            (body_word_count - AVG(body_word_count) OVER (PARTITION BY detected_language))
                / NULLIF(STDDEV(body_word_count) OVER (PARTITION BY detected_language), 0)
                AS body_lang_zscore,
            (body_word_count - AVG(body_word_count) OVER (PARTITION BY product_category_id))
                / NULLIF(STDDEV(body_word_count) OVER (PARTITION BY product_category_id), 0)
                AS body_cat_zscore,
            headline_word_count * 1.0
                / NULLIF(body_word_count, 0)                         AS headline_body_ratio,
            exclamation_count * 1.0
                / NULLIF(sentence_count_approx, 0)                   AS exclamation_density,
            question_count * 1.0
                / NULLIF(sentence_count_approx, 0)                   AS question_density
        FROM text_stats
    )
    SELECT * FROM lang_norms
""")

con.execute("""
    SELECT label, COUNT(*) AS reviews,
        ROUND(AVG(body_word_count), 1)    AS avg_words,
        ROUND(AVG(type_token_ratio), 3)   AS avg_ttr,
        ROUND(AVG(body_lang_zscore), 3)   AS avg_lang_zscore,
        ROUND(AVG(exclamation_density), 4) AS avg_excl_density
    FROM gold.review_lexical_features
    WHERE label IS NOT NULL
    GROUP BY label ORDER BY label DESC
""").df()

<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
C:\Users\user\AppData\Local\Temp\ipykernel_12388\2460651053.py:4: SyntaxWarning: invalid escape sequence '\s'
  con.execute("""


,label,reviews,avg_words,avg_ttr,avg_lang_zscore,avg_excl_density
0,True,4474,136.3,0.804,0.326,0.4010
1,False,5139,46.4,0.889,-0.235,0.3692


## Embedding Features

In [6]:
MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(MODEL_NAME)
print(f'Embedding dim:  {model.get_sentence_embedding_dimension()}')
print(f'Max seq length: {model.max_seq_length} tokens')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15683.03it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dim:  384
Max seq length: 128 tokens


In [7]:
def encode_field(model, series, batch_size=64, field_name=''):
    texts = series.fillna('').str.strip()
    valid_mask = (texts != '').values
    inputs = texts.where(texts != '', other=' ').tolist()
    print(f'Encoding {field_name} ({valid_mask.sum():,} / {len(texts):,} non-empty) ...')
    embeddings = model.encode(inputs, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=False)
    return embeddings.astype(np.float32), valid_mask

def cosine_sim_rowwise(a, b, mask_a, mask_b):
    a_norm = a / np.clip(np.linalg.norm(a, axis=1, keepdims=True), 1e-8, None)
    b_norm = b / np.clip(np.linalg.norm(b, axis=1, keepdims=True), 1e-8, None)
    sim = np.einsum('ij,ij->i', a_norm, b_norm).astype(float)
    sim[~(mask_a & mask_b)] = np.nan
    return sim

body_emb,     body_valid     = encode_field(model, silver_df['review_body'],     field_name='review_body')
headline_emb, headline_valid = encode_field(model, silver_df['review_headline'], field_name='review_headline')
title_emb,    title_valid    = encode_field(model, silver_df['product_title'],   field_name='product_title')

silver_df['headline_body_cosine_sim'] = cosine_sim_rowwise(headline_emb, body_emb, headline_valid, body_valid)
silver_df['title_body_cosine_sim']    = cosine_sim_rowwise(title_emb, body_emb, title_valid, body_valid)
body_norms = np.linalg.norm(body_emb, axis=1).astype(float)
body_norms[~body_valid] = np.nan
silver_df['body_embedding_norm'] = body_norms

print('\nSimilarity feature stats:')
silver_df[['headline_body_cosine_sim', 'title_body_cosine_sim', 'body_embedding_norm']].describe().round(3)

Encoding review_body (11,999 / 11,999 non-empty) ...


Batches: 100%|██████████| 188/188 [01:56<00:00,  1.61it/s]


Encoding review_headline (11,087 / 11,999 non-empty) ...


Batches: 100%|██████████| 188/188 [00:20<00:00,  9.34it/s]


Encoding product_title (11,910 / 11,999 non-empty) ...


Batches: 100%|██████████| 188/188 [00:24<00:00,  7.58it/s]



Similarity feature stats:


,headline_body_cosine_sim,title_body_cosine_sim,body_embedding_norm
count,11087.000,11910.000,11999.000
mean,0.327,0.287,3.432
std,0.185,0.173,0.704
min,-0.145,-0.124,1.633
25%,0.190,0.159,2.960
50%,0.297,0.264,3.301
75%,0.441,0.398,3.783
max,1.000,0.964,6.513


In [8]:
emb_features_df = silver_df[['_index', 'product_id', 'label',
                               'headline_body_cosine_sim', 'title_body_cosine_sim',
                               'body_embedding_norm']].copy()

con.register('embedding_features_temp', emb_features_df)
con.execute('DROP TABLE IF EXISTS gold.review_embedding_features')
con.execute('CREATE TABLE gold.review_embedding_features AS SELECT * FROM embedding_features_temp')

con.execute("""
    SELECT label, COUNT(*) AS reviews,
        ROUND(AVG(headline_body_cosine_sim), 4) AS avg_headline_body_sim,
        ROUND(AVG(title_body_cosine_sim),    4) AS avg_title_body_sim,
        ROUND(AVG(body_embedding_norm),      3) AS avg_body_norm
    FROM gold.review_embedding_features
    WHERE label IS NOT NULL
    GROUP BY label ORDER BY label DESC
""").df()

,label,reviews,avg_headline_body_sim,avg_title_body_sim,avg_body_norm
0,True,4474,0.3289,0.3035,3.186
1,False,5139,0.3250,0.2744,3.652


In [9]:
from sklearn.decomposition import PCA

N_COMPONENTS = 15
pca_model = PCA(n_components=N_COMPONENTS, random_state=42)
body_pca   = pca_model.fit_transform(body_emb)  
explained_var = pca_model.explained_variance_ratio_.cumsum()

pca_col_names = [f'body_emb_pca_{i}' for i in range(N_COMPONENTS)]
pca_df = pd.DataFrame(body_pca, columns=pca_col_names)
pca_df['_index']     = silver_df['_index'].values
pca_df['product_id'] = silver_df['product_id'].values
pca_df['label']      = silver_df['label'].values

con.register('pca_temp', pca_df)
con.execute('DROP TABLE IF EXISTS gold.review_embedding_pca')
con.execute('CREATE TABLE gold.review_embedding_pca AS SELECT * FROM pca_temp')

check = con.execute("""
    SELECT label,
        ROUND(AVG(body_emb_pca_0), 4) AS avg_pc0,
        ROUND(AVG(body_emb_pca_1), 4) AS avg_pc1,
        ROUND(AVG(body_emb_pca_2), 4) AS avg_pc2,
        ROUND(STDDEV(body_emb_pca_0), 4) AS std_pc0
    FROM gold.review_embedding_pca
    WHERE label IS NOT NULL
    GROUP BY label ORDER BY label DESC
""").df()
print("\nPCA component means by label:")
print(check.to_string(index=False))
print(f"\ngold.review_embedding_pca: {len(pca_df):,} rows x {N_COMPONENTS} components")



PCA component means by label:
 label  avg_pc0  avg_pc1  avg_pc2  std_pc0
  True   0.0141  -0.2578  -0.0264   0.8331
 False  -0.0279   0.2417   0.0318   0.9096

gold.review_embedding_pca: 11,999 rows x 15 components


## Sentiment & Quality Features


In [10]:
SENTIMENT_MODEL = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
sentiment_pipe = pipeline(
    'text-classification',
    model=SENTIMENT_MODEL,
    top_k=None,
    truncation=True,
    max_length=256,
    device=-1,
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 59282.41it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
def score_sentiment(texts, batch_size=32):
    clean = [str(t).strip() if t and str(t).strip() else ' ' for t in texts]
    valid = [t != ' ' for t in clean]
    results = sentiment_pipe(clean, batch_size=batch_size)
    rows = []
    for res, is_valid in zip(results, valid):
        if not is_valid:
            rows.append({'label': None, 'score': None, 'polarity': None})
            continue
        scores = {r['label'].lower(): r['score'] for r in res}
        pos = scores.get('positive', 0.0)
        neg = scores.get('negative', 0.0)
        top = max(res, key=lambda x: x['score'])
        rows.append({'label': top['label'], 'score': top['score'], 'polarity': float(pos - neg)})
    return pd.DataFrame(rows)

body_sent = score_sentiment(silver_df['review_body'].tolist())
silver_df['body_sentiment_label']    = body_sent['label'].values
silver_df['body_sentiment_score']    = body_sent['score'].values
silver_df['body_sentiment_polarity'] = body_sent['polarity'].values


headline_sent = score_sentiment(silver_df['review_headline'].tolist())
silver_df['headline_sentiment_label']    = headline_sent['label'].values
silver_df['headline_sentiment_score']    = headline_sent['score'].values
silver_df['headline_sentiment_polarity'] = headline_sent['polarity'].values

silver_df['sentiment_mismatch'] = np.where(
    silver_df['body_sentiment_polarity'].isna() | silver_df['headline_sentiment_polarity'].isna(),
    np.nan,
    np.abs(silver_df['body_sentiment_polarity'] - silver_df['headline_sentiment_polarity'])
)
print('Body sentiment distribution:')
print(silver_df['body_sentiment_label'].value_counts())
print('\nPolarity stats:')
print(silver_df['body_sentiment_polarity'].describe().round(3))

Body sentiment distribution:
body_sentiment_label
positive    7542
negative    2832
neutral     1625
Name: count, dtype: int64

Polarity stats:
count    11999.000
mean         0.329
std          0.575
min         -0.960
25%         -0.122
50%          0.523
75%          0.873
max          0.950
Name: body_sentiment_polarity, dtype: float64


In [12]:
TEXTSTAT_LANG_MAP = {'en': 'en', 'de': 'de', 'fr': 'fr', 'es': 'es', 'it': 'it', 'nl': 'nl', 'ru': 'ru'}

def repetition_ratio(text):
    if not text or not str(text).strip():
        return None
    sentences = re.split(r'[.!?]+', str(text))
    sentences = [s.strip().lower() for s in sentences if s.strip()]
    if len(sentences) <= 1:
        return 0.0
    return (len(sentences) - len(set(sentences))) / len(sentences)

def flesch_ease(row):
    lang = TEXTSTAT_LANG_MAP.get(row.get('detected_language', ''), 'en')
    text = row.get('review_body', '') or ''
    if not text.strip():
        return None
    textstat.set_lang(lang)
    try:
        return float(textstat.flesch_reading_ease(text))
    except Exception:
        return None

def jaccard_sim(text_a, text_b):
    if not text_a or not text_b:
        return None
    a, b = set(str(text_a).lower().split()), set(str(text_b).lower().split())
    union = len(a | b)
    return len(a & b) / union if union > 0 else 0.0

def overlap_ratio(text_source, text_target):
    if not text_source or not text_target:
        return None
    src, tgt = set(str(text_source).lower().split()), set(str(text_target).lower().split())
    return len(src & tgt) / len(src) if src else None

def bigram_diversity(text):
    if not text or not str(text).strip():
        return None
    tokens = str(text).lower().split()
    if len(tokens) < 2:
        return None
    bigrams = list(zip(tokens, tokens[1:]))
    return len(set(bigrams)) / len(bigrams)

silver_df['repetition_ratio']      = silver_df['review_body'].apply(repetition_ratio)
silver_df['flesch_reading_ease']   = silver_df[['review_body', 'detected_language']].apply(flesch_ease, axis=1)
silver_df['headline_body_jaccard'] = silver_df.apply(lambda r: jaccard_sim(r['review_headline'], r['review_body']), axis=1)
silver_df['title_body_jaccard']    = silver_df.apply(lambda r: jaccard_sim(r['product_title'], r['review_body']), axis=1)
silver_df['title_body_overlap']    = silver_df.apply(lambda r: overlap_ratio(r['product_title'], r['review_body']), axis=1)
silver_df['body_bigram_diversity'] = silver_df['review_body'].apply(bigram_diversity)


In [13]:
con.register('silver_enriched', silver_df)

con.execute('DROP TABLE IF EXISTS gold.review_sentiment_features')
con.execute("""
    CREATE TABLE gold.review_sentiment_features AS
    WITH base AS (
        SELECT
            _index, product_id, product_parent, label,
            detected_language, product_category_id,
            body_sentiment_label, body_sentiment_score, body_sentiment_polarity,
            headline_sentiment_label, headline_sentiment_score, headline_sentiment_polarity,
            sentiment_mismatch, repetition_ratio, flesch_reading_ease,
            headline_body_jaccard, title_body_jaccard, title_body_overlap, body_bigram_diversity,
            COALESCE(ARRAY_LENGTH(regexp_extract_all(review_body, '[A-Z]')), 0) * 1.0
                / NULLIF(COALESCE(ARRAY_LENGTH(regexp_extract_all(review_body, '[a-zA-Z]')), 0), 0) AS uppercase_ratio,
            COALESCE(ARRAY_LENGTH(regexp_extract_all(review_body, '[0-9]')), 0) * 1.0
                / NULLIF(ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' ')), 0) AS digit_density,
            GREATEST(COALESCE(ARRAY_LENGTH(regexp_extract_all(review_body, '[.!?]+')), 0), 1) AS sentence_count,
            ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' ')) * 1.0
                / NULLIF(GREATEST(COALESCE(ARRAY_LENGTH(regexp_extract_all(review_body, '[.!?]+')), 0), 1), 0) AS avg_sentence_length
        FROM silver_enriched
        WHERE review_body IS NOT NULL AND TRIM(review_body) != ''
    ),
    normed AS (
        SELECT *,
            (flesch_reading_ease - AVG(flesch_reading_ease) OVER (PARTITION BY detected_language))
                / NULLIF(STDDEV(flesch_reading_ease) OVER (PARTITION BY detected_language), 0)
                AS flesch_lang_zscore,
            (flesch_reading_ease - AVG(flesch_reading_ease) OVER (PARTITION BY product_category_id))
                / NULLIF(STDDEV(flesch_reading_ease) OVER (PARTITION BY product_category_id), 0)
                AS flesch_cat_zscore,
            (body_sentiment_polarity - AVG(body_sentiment_polarity) OVER (PARTITION BY detected_language))
                / NULLIF(STDDEV(body_sentiment_polarity) OVER (PARTITION BY detected_language), 0)
                AS polarity_lang_zscore
        FROM base
    )
    SELECT * FROM normed
""")

con.execute("""
    SELECT label, COUNT(*) AS reviews,
        ROUND(AVG(body_sentiment_polarity), 3)  AS avg_polarity,
        ROUND(AVG(sentiment_mismatch), 3)       AS avg_mismatch,
        ROUND(AVG(flesch_reading_ease), 2)      AS avg_flesch,
        ROUND(AVG(repetition_ratio), 4)         AS avg_repetition,
        ROUND(AVG(body_bigram_diversity), 3)    AS avg_bigram_diversity
    FROM gold.review_sentiment_features
    WHERE label IS NOT NULL
    GROUP BY label ORDER BY label DESC
""").df()

,label,reviews,avg_polarity,avg_mismatch,avg_flesch,avg_repetition,avg_bigram_diversity
0,True,4474,0.160,0.503,73.32,0.0063,0.977
1,False,5139,0.472,0.514,75.80,0.0010,0.988


## Evidence Features

In [14]:
con.execute("CREATE OR REPLACE TEMP TABLE silver_temp AS SELECT * FROM silver_with_lang")
con.execute('DROP TABLE IF EXISTS gold.review_evidence_features')
con.execute("""
    CREATE TABLE gold.review_evidence_features AS
    WITH base AS (
        SELECT
            _index, product_id, product_parent, label, product_category_id, review_body,
            GREATEST(COALESCE(ARRAY_LENGTH(regexp_extract_all(review_body, '[.!?]+')), 0), 1) AS sentence_count_approx,
            ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' ')) AS word_count,
            COALESCE(ARRAY_LENGTH(regexp_extract_all(
                review_body,
                '\d+[\.,]?\d*\s{0,2}(cm|mm|km|mg|kg|gb|mb|tb|ml|cl|dl|hz|khz|mhz|ghz|mp|fps|dpi|ppi|rpm|watt|wh|mah|lm|db|nm|ft|inch|oz|lb|yd|Р’В°c|Р’В°f|Р’В°|%|p\b|h\b|min\b|sec\b|g\b|l\b|m\b|k\b)'
            )), 0) AS measurement_count,
            COALESCE(ARRAY_LENGTH(regexp_extract_all(
                review_body,
                '[$РІвЂљВ¬Р’Р€Р’Тђ]\s?\d+[\.,]?\d*|\d+[\.,]?\d*\s?[$РІвЂљВ¬Р’Р€Р’Тђ]'
            )), 0) AS price_ref_count,
            COALESCE(ARRAY_LENGTH(regexp_extract_all(
                review_body,
                '\b(best|worst|better|worse|compared to|versus|vs\.?|unlike|superior|inferior|first|second|third|top|bottom|highest|lowest|greatest|least|most|fewer|more than|less than)\b'
            )), 0) AS ordinal_comparison_count
        FROM silver_temp
        WHERE review_body IS NOT NULL AND TRIM(review_body) != ''
    ),
    densities AS (
        SELECT
            _index, product_id, product_parent, label, product_category_id,
            measurement_count, price_ref_count, ordinal_comparison_count,
            measurement_count    * 1.0 / sentence_count_approx AS measurement_density,
            price_ref_count      * 1.0 / sentence_count_approx AS price_reference_density,
            ordinal_comparison_count * 1.0 / sentence_count_approx AS ordinal_comparison_density,
            (3.0 * measurement_count + 2.0 * price_ref_count + 1.0 * ordinal_comparison_count)
                * 1.0 / NULLIF(sentence_count_approx, 0) AS quantitative_evidence_score
        FROM base
    ),
    normed AS (
        SELECT *,
            (quantitative_evidence_score
                - AVG(quantitative_evidence_score) OVER (PARTITION BY product_category_id))
                / NULLIF(STDDEV(quantitative_evidence_score) OVER (PARTITION BY product_category_id), 0)
                AS evidence_cat_zscore
        FROM densities
    )
    SELECT * FROM normed
""")

con.execute("""
    SELECT label, COUNT(*) AS reviews,
        ROUND(AVG(measurement_density),          4) AS avg_meas_density,
        ROUND(AVG(price_reference_density),      4) AS avg_price_density,
        ROUND(AVG(quantitative_evidence_score),  4) AS avg_evidence_score,
        ROUND(AVG(evidence_cat_zscore),          3) AS avg_evidence_zscore
    FROM gold.review_evidence_features
    WHERE label IS NOT NULL
    GROUP BY label ORDER BY label DESC
""").df()

<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
C:\Users\user\AppData\Local\Temp\ipykernel_12388\2103800041.py:3: SyntaxWarning: invalid escape sequence '\d'
  con.execute("""


,label,reviews,avg_meas_density,avg_price_density,avg_evidence_score,avg_evidence_zscore
0,True,4474,0.0029,0.0028,0.0141,0.015
1,False,5139,0.0044,0.0008,0.0148,-0.017


## Novelty Features

1. product_tfidf_novelty - Generic reviews score near 0. Detailed, specific reviews score high.

2. product_vocab_expansion_ratio - If most reviews have already been written about a product and the one/some uses entirely new vocabulary, it's likely adding new genuine information.

3. product_review_rank_novelty - A score of 0.9 means this review is more unique than 90% of reviews for the same product

In [15]:
def compute_novelty_for_product(group_df):
    n = len(group_df)
    indices = group_df.index.tolist()
    texts = group_df['review_body'].fillna('').tolist()

    novelty = np.full(n, np.nan)
    vocab_expansion = np.full(n, np.nan)

    if n >= 2:
        try:
            vec = TfidfVectorizer(min_df=1, sublinear_tf=True)
            tfidf_matrix = vec.fit_transform(texts)
            # Convert to dense array to avoid numpy.matrix deprecation issues
            total = np.asarray(tfidf_matrix.sum(axis=0))  # shape (1, vocab)
            for i in range(n):
                row_i = np.asarray(tfidf_matrix[i].todense())  # shape (1, vocab)
                centroid = (total - row_i) / (n - 1)
                novelty[i] = 1.0 - float(cosine_similarity(row_i, centroid)[0, 0])
        except Exception as e:
            pass

    seen_tokens = set()
    for i, text in enumerate(texts):
        tokens = set(str(text).lower().split())
        if i == 0:
            seen_tokens = tokens
        else:
            new_tokens = tokens - seen_tokens
            vocab_expansion[i] = len(new_tokens) / len(tokens) if tokens else np.nan
            seen_tokens |= tokens

    return pd.DataFrame({
        '_row_pos': indices,
        'product_tfidf_novelty': novelty,
        'product_vocab_expansion_ratio': vocab_expansion,
    })


novelty_base = silver_df[['_index', 'product_id', 'product_parent', 'label',
                           'product_category_id', 'review_date', 'review_body']].copy()
novelty_base = novelty_base.sort_values(['product_parent', 'review_date', '_index']).reset_index(drop=False)
novelty_base = novelty_base.rename(columns={'index': '_row_pos'})

result_parts = []
for pid, group in novelty_base.groupby('product_parent', sort=False):
    result_parts.append(compute_novelty_for_product(group))

novelty_df = pd.concat(result_parts, ignore_index=True)
novelty_merged = novelty_base.merge(novelty_df, on='_row_pos')

print(f'Novelty NaN rate: {novelty_merged["product_tfidf_novelty"].isna().mean():.2%}')
print(f'Vocab expansion NaN rate: {novelty_merged["product_vocab_expansion_ratio"].isna().mean():.2%}')
print(f'\nNovelty stats:')
print(novelty_merged['product_tfidf_novelty'].describe().round(3))

Novelty NaN rate: 26.67%
Vocab expansion NaN rate: 46.86%

Novelty stats:
count    8799.000
mean        0.895
std         0.092
min         0.361
25%         0.832
50%         0.911
75%         0.977
max         1.000
Name: product_tfidf_novelty, dtype: float64


In [16]:
con.register('novelty_temp', novelty_merged)

con.execute('DROP TABLE IF EXISTS gold.review_novelty_features')
con.execute("""
    CREATE TABLE gold.review_novelty_features AS
    SELECT
        _index, product_id, product_parent, label,
        product_tfidf_novelty,
        product_vocab_expansion_ratio,
        PERCENT_RANK() OVER (
            PARTITION BY product_parent
            ORDER BY product_tfidf_novelty NULLS FIRST
        ) AS product_review_rank_novelty
    FROM novelty_temp
    WHERE _index IS NOT NULL
""")

con.execute("""
    SELECT label, COUNT(*) AS reviews,
        ROUND(AVG(product_tfidf_novelty),         3) AS avg_novelty,
        ROUND(AVG(product_vocab_expansion_ratio), 3) AS avg_vocab_expansion,
        ROUND(AVG(product_review_rank_novelty),   3) AS avg_rank_novelty
    FROM gold.review_novelty_features
    WHERE label IS NOT NULL
    GROUP BY label ORDER BY label DESC
""").df()

,label,reviews,avg_novelty,avg_vocab_expansion,avg_rank_novelty
0,True,4474,0.895,0.802,0.320
1,False,5139,0.895,0.800,0.351


## Product Context Features

In [17]:
con.execute('DROP TABLE IF EXISTS gold.review_product_context')
con.execute("""
    CREATE TABLE gold.review_product_context AS
    WITH base AS (
        SELECT
            _index, product_id, product_parent, label,
            vine, verified_purchase, review_date, marketplace_id, product_category_id,
            COUNT(*) OVER (PARTITION BY product_parent)           AS product_review_count,
            ROW_NUMBER() OVER (
                PARTITION BY product_parent ORDER BY review_date
            )                                                     AS review_relative_rank,
            MIN(review_date) OVER (PARTITION BY product_parent)   AS first_review_date
        FROM silver_data
    ),
    with_early AS (
        SELECT *,
            CASE
                WHEN review_relative_rank <= CEIL(product_review_count * 0.10)
                THEN TRUE ELSE FALSE
            END AS is_early_review,
            DATE_DIFF('day', first_review_date, review_date) AS days_since_first_review
        FROM base
    ),
    with_category_stats AS (
        SELECT *,
            PERCENT_RANK() OVER (
                PARTITION BY product_category_id ORDER BY product_review_count
            ) AS product_popularity_pctile,
            AVG(product_review_count) OVER (
                PARTITION BY product_category_id
            ) AS category_review_density,
            COUNT(DISTINCT product_parent) OVER (
                PARTITION BY product_category_id
            ) AS category_product_count
        FROM with_early
    )
    SELECT
        _index, product_id, product_parent, label, product_category_id,
        product_review_count,
        product_popularity_pctile,
        ROUND(category_review_density, 2)                                      AS category_review_density,
        category_product_count,
        (is_early_review AND product_popularity_pctile > 0.75)                 AS early_in_popular_product,
        (vine = 'Y' AND product_review_count < 10)                             AS vine_in_sparse_product,
        days_since_first_review * product_popularity_pctile                    AS days_since_first_x_popularity,
        (verified_purchase = 'Y' AND product_popularity_pctile > 0.75)         AS verified_in_popular_product,
        ROUND(product_popularity_pctile * product_review_count, 2)             AS popularity_weight
    FROM with_category_stats
""")


con.execute("""
    SELECT label, COUNT(*) AS reviews,
        ROUND(AVG(product_popularity_pctile),      3) AS avg_pop_pctile,
        ROUND(AVG(category_review_density),        2) AS avg_cat_density,
        SUM(CASE WHEN early_in_popular_product  THEN 1 ELSE 0 END) AS early_in_popular,
        SUM(CASE WHEN vine_in_sparse_product    THEN 1 ELSE 0 END) AS vine_sparse,
        ROUND(AVG(popularity_weight),              2) AS avg_pop_weight
    FROM gold.review_product_context
    WHERE label IS NOT NULL
    GROUP BY label ORDER BY label DESC
""").df()

,label,reviews,avg_pop_pctile,avg_cat_density,early_in_popular,vine_sparse,avg_pop_weight
0,True,4474,0.385,5.12,224.0,10.0,3.29
1,False,5139,0.447,5.85,115.0,7.0,4.65


## Similarity, Specificity & Length Features

In [18]:
con.execute('DROP TABLE IF EXISTS gold.review_similarity_features')

In [19]:
con.execute('DROP TABLE IF EXISTS gold.review_specificity_features')
con.execute("""
    CREATE TABLE gold.review_specificity_features AS
    WITH base AS (
        SELECT
            _index, product_id, product_parent, label,
            ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' '))  AS word_count,

            -- First-person pronouns: marker of personal genuine experience
            COALESCE(ARRAY_LENGTH(regexp_extract_all(
                review_body,
                '(?i)\\b(I|my|me|myself|mine)\\b'
            )), 0) AS first_person_count,

            -- Superlatives & extreme adjectives: over-used in fake reviews
            COALESCE(ARRAY_LENGTH(regexp_extract_all(
                review_body,
                '(?i)\\b(best|worst|amazing|perfect|terrible|awesome|horrible|excellent|awful|fantastic|outstanding|incredible|useless|pathetic|brilliant|superb|dreadful|magnificent|exceptional|phenomenal|extraordinary|flawless|garbage|disgusting|wonderful|breathtaking|atrocious)\\b'
            )), 0) AS superlative_count,

            -- Numbers/measurements: marker of specific factual content
            COALESCE(ARRAY_LENGTH(regexp_extract_all(
                review_body,
                '\\b\\d+[.,]?\\d*\\s{0,2}(cm|mm|km|mg|kg|gb|mb|tb|ml|hz|mp|fps|\\$|РІвЂљВ¬|Р’Р€|%|inch|oz|lb)\\b'
            )), 0) AS specific_detail_count,

            -- Review length percentile within the same product
            PERCENT_RANK() OVER (
                PARTITION BY product_parent
                ORDER BY ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' '))
            ) AS body_word_count_pctile_in_product,

            -- Review length relative to product average
            ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' ')) * 1.0
                / NULLIF(AVG(ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' ')))
                         OVER (PARTITION BY product_parent), 0)
                AS word_count_vs_product_avg

        FROM silver_temp
        WHERE review_body IS NOT NULL AND TRIM(review_body) != ''
    )
    SELECT
        _index, product_id, product_parent, label,
        first_person_count,
        superlative_count,
        specific_detail_count,

        -- Densities (per word)
        first_person_count   * 1.0 / NULLIF(word_count, 0) AS first_person_density,
        superlative_count    * 1.0 / NULLIF(word_count, 0) AS superlative_density,
        specific_detail_count * 1.0 / NULLIF(word_count, 0) AS specific_detail_density,

        -- Specificity score: high personal + high detail - high superlatives = genuine
        (first_person_count + specific_detail_count - superlative_count)
            * 1.0 / NULLIF(word_count, 0)                  AS specificity_score,

        body_word_count_pctile_in_product,
        word_count_vs_product_avg
    FROM base
""")

con.execute("""
    SELECT label, COUNT(*) AS reviews,
        ROUND(AVG(first_person_density),    4) AS avg_first_person,
        ROUND(AVG(superlative_density),     4) AS avg_superlative,
        ROUND(AVG(specificity_score),       4) AS avg_specificity,
        ROUND(AVG(body_word_count_pctile_in_product), 3) AS avg_length_pctile
    FROM gold.review_specificity_features
    WHERE label IS NOT NULL
    GROUP BY label ORDER BY label DESC
""").df()

,label,reviews,avg_first_person,avg_superlative,avg_specificity,avg_length_pctile
0,True,4474,0.0113,0.0033,0.0081,0.433
1,False,5139,0.0173,0.0189,-0.0015,0.298


### Advanced Features: Burst Detection, Vocabulary Overlap, Character Signals, Sentiment Arc, Title Coverage, Language Z-scores

In [20]:
from collections import Counter
import unicodedata

n      = len(silver_df)
bodies = silver_df['review_body'].fillna('').tolist()
titles = silver_df['product_title'].fillna('').tolist()

print("1/6 Vocabulary overlap...")
def tokenize_set(text):
    return set(re.findall(r'\b[a-z]{2,}\b', text.lower())) if isinstance(text, str) else set()

token_sets  = [tokenize_set(t) for t in bodies]
prod_groups = silver_df.groupby('product_parent', sort=False)['_index'].apply(list).to_dict()

vocab_overlap = np.full(n, np.nan)
for indices in prod_groups.values():
    if len(indices) < 2:
        continue
    for i in indices:
        my = token_sets[i]
        if not my:
            continue
        others = set().union(*(token_sets[j] for j in indices if j != i))
        vocab_overlap[i] = len(my & others) / len(my)
print(f"   NaN={np.isnan(vocab_overlap).mean():.1%}  mean={np.nanmean(vocab_overlap):.3f}")

print("2/6 Character n-gram entropy...")
def char_ngram_entropy(text, ng=3):
    if not isinstance(text, str) or len(text) < ng:
        return np.nan
    s      = text[:1000]
    ngrams = [s[i:i+ng] for i in range(len(s) - ng + 1)]
    counts = Counter(ngrams)
    total  = sum(counts.values())
    probs  = np.array([c / total for c in counts.values()])
    return float(-np.sum(probs * np.log2(probs + 1e-10)))

char_entropy = np.array([char_ngram_entropy(t) for t in bodies])
print(f"   mean={np.nanmean(char_entropy):.3f}  std={np.nanstd(char_entropy):.3f}")

print("3/6 Script homogeneity...")
def script_homogeneity(text):
    if not isinstance(text, str) or not text:
        return np.nan
    scripts = Counter()
    for ch in text[:500]:
        if unicodedata.category(ch).startswith('L'):
            name = unicodedata.name(ch, 'UNKNOWN')
            scripts[name.split()[0]] += 1
    if not scripts:
        return np.nan
    dominant = scripts.most_common(1)[0][1]
    return dominant / sum(scripts.values())

script_homo = np.array([script_homogeneity(t) for t in bodies])
print(f"   mean={np.nanmean(script_homo):.3f}")

print("4/6 Review burst (Р’В±3 days)...")
sf = silver_df[['_index', 'product_parent', 'review_date']].copy()
sf['review_date_d'] = pd.to_datetime(sf['review_date']).dt.normalize()
burst = np.zeros(n, dtype=np.int32)
for prod, grp in sf.groupby('product_parent', sort=False):
    if len(grp) < 2:
        continue
    dates = grp['review_date_d'].values.astype('datetime64[D]').astype(np.int64)
    idxs  = grp['_index'].values
    for i in range(len(dates)):
        burst[idxs[i]] = int(np.sum(np.abs(dates - dates[i]) <= 3)) - 1
print(f"   mean={burst.mean():.2f}  max={burst.max()}")

print("5/6 Title word coverage...")
def title_coverage(title, body):
    if not isinstance(title, str) or not isinstance(body, str):
        return np.nan
    tw = set(re.findall(r'\b[a-z]{2,}\b', title.lower()))
    bw = set(re.findall(r'\b[a-z]{2,}\b', body.lower()))
    return len(tw & bw) / len(tw) if tw else np.nan

title_cov = np.array([title_coverage(t, b) for t, b in zip(titles, bodies)])
print(f"   NaN={np.isnan(title_cov).mean():.1%}  mean={np.nanmean(title_cov):.3f}")

print("6/6 Sentiment arc (running model on body halves РІР‚вЂќ takes a few minutes)...")

def split_halves(text):
    if not isinstance(text, str) or not text.strip():
        return ' ', ' '
    words = text.split()
    mid   = max(1, len(words) // 2)
    return ' '.join(words[:mid]), ' '.join(words[mid:])

first_halves  = [split_halves(t)[0] for t in bodies]
second_halves = [split_halves(t)[1] for t in bodies]

def batch_polarity(texts, batch_size=64):
    out = np.zeros(len(texts))
    for i in range(0, len(texts), batch_size):
        batch = [t if t.strip() else ' ' for t in texts[i:i+batch_size]]
        try:
            res = sentiment_pipe(batch, truncation=True, max_length=128)
            for j, r_list in enumerate(res):
                sc = {r['label'].lower(): r['score'] for r in r_list}
                out[i+j] = sc.get('positive', 0.0) - sc.get('negative', 0.0)
        except Exception:
            pass
    return out

first_pol  = batch_polarity(first_halves)
second_pol = batch_polarity(second_halves)
sentiment_arc = second_pol - first_pol
print(f"   arc mean={sentiment_arc.mean():.3f}  std={sentiment_arc.std():.3f}")

silver_df['vocab_overlap_with_product'] = vocab_overlap
silver_df['char_ngram_entropy']          = char_entropy
silver_df['script_homogeneity']          = script_homo
silver_df['reviews_in_3day_window']      = burst
silver_df['title_word_coverage']         = title_cov
silver_df['first_half_sentiment']        = first_pol
silver_df['second_half_sentiment']       = second_pol
silver_df['sentiment_arc']               = sentiment_arc
print("\nAll 6 Python advanced features stored in silver_df.")

1/6 Vocabulary overlap...
   NaN=26.9%  mean=0.273
2/6 Character n-gram entropy...
   mean=7.352  std=1.328
3/6 Script homogeneity...
   mean=1.000
4/6 Review burst (Р’В±3 days)...
   mean=0.08  max=10
5/6 Title word coverage...
   NaN=2.1%  mean=0.219
6/6 Sentiment arc (running model on body halves РІР‚вЂќ takes a few minutes)...
   arc mean=-0.043  std=0.636

All 6 Python advanced features stored in silver_df.


In [21]:
adv_py = silver_df[['_index', 'product_id', 'product_parent', 'label',
                     'vocab_overlap_with_product', 'char_ngram_entropy',
                     'script_homogeneity', 'reviews_in_3day_window',
                     'title_word_coverage', 'first_half_sentiment',
                     'second_half_sentiment', 'sentiment_arc']].copy()
con.register('adv_py_temp', adv_py)

con.execute('DROP TABLE IF EXISTS gold.review_advanced_features')
con.execute("""
    CREATE TABLE gold.review_advanced_features AS
    WITH sentiment_dev AS (
        SELECT
            sm._index,
            sm.body_sentiment_score
                - MEDIAN(sm.body_sentiment_score) OVER (
                    PARTITION BY py.product_parent
                  ) AS sentiment_product_deviation,
            sm.body_sentiment_score
                - MEDIAN(sm.body_sentiment_score) OVER (
                    PARTITION BY lx.detected_language
                  ) AS sentiment_lang_deviation
        FROM gold.review_sentiment_features sm
        JOIN adv_py_temp              py ON py._index = sm._index
        JOIN gold.review_lexical_features lx ON lx._index = sm._index
    ),
    lang_zscores AS (
        SELECT
            lx._index,
            (lx.avg_word_length - AVG(lx.avg_word_length) OVER (PARTITION BY lx.detected_language))
                / NULLIF(STDDEV(lx.avg_word_length) OVER (PARTITION BY lx.detected_language), 0)
                AS avg_word_length_lang_zscore,
            (lx.type_token_ratio - AVG(lx.type_token_ratio) OVER (PARTITION BY lx.detected_language))
                / NULLIF(STDDEV(lx.type_token_ratio) OVER (PARTITION BY lx.detected_language), 0)
                AS ttr_lang_zscore,
            (sm.body_sentiment_score - AVG(sm.body_sentiment_score) OVER (PARTITION BY lx.detected_language))
                / NULLIF(STDDEV(sm.body_sentiment_score) OVER (PARTITION BY lx.detected_language), 0)
                AS sentiment_score_lang_zscore
        FROM gold.review_lexical_features lx
        JOIN gold.review_sentiment_features sm ON sm._index = lx._index
    )
    SELECT
        py._index, py.product_id, py.product_parent, py.label,
        py.vocab_overlap_with_product,
        py.char_ngram_entropy,
        py.script_homogeneity,
        py.reviews_in_3day_window,
        py.title_word_coverage,
        py.first_half_sentiment,
        py.second_half_sentiment,
        py.sentiment_arc,
        sd.sentiment_product_deviation,
        sd.sentiment_lang_deviation,
        lz.avg_word_length_lang_zscore,
        lz.ttr_lang_zscore,
        lz.sentiment_score_lang_zscore
    FROM adv_py_temp py
    LEFT JOIN sentiment_dev  sd ON sd._index = py._index
    LEFT JOIN lang_zscores   lz ON lz._index = py._index
""")

summary = con.execute("""
    SELECT
        ROUND(AVG(char_ngram_entropy),         3) AS avg_char_entropy,
        ROUND(AVG(vocab_overlap_with_product), 3) AS avg_vocab_overlap,
        ROUND(AVG(ABS(sentiment_arc)),         3) AS avg_arc_magnitude
    FROM gold.review_advanced_features
""").df()
print("Advanced features summary:")
print(summary.to_string(index=False))

n_rows = con.execute('SELECT COUNT(*) FROM gold.review_advanced_features').fetchone()[0]
n_cols = con.execute("SELECT COUNT(*) FROM pragma_table_info('gold.review_advanced_features')").fetchone()[0]
print(f"\ngold.review_advanced_features: {n_rows:,} rows x {n_cols} columns")

Advanced features summary:
 avg_char_entropy  avg_vocab_overlap  avg_arc_magnitude
            7.352              0.273              0.442

gold.review_advanced_features: 11,999 rows x 17 columns


## Join all features on `_index` into a single wide table

In [22]:
con.execute('DROP TABLE IF EXISTS gold.ml_features')
con.execute("""
    CREATE TABLE gold.ml_features AS
    SELECT
        -- Identity & metadata
        t._index, t.product_id, t.product_parent, t.label, t.dataset_split,
        t.marketplace_id, t.product_category_id,
        t.vine, t.verified_purchase, t.review_date,
        lx.detected_language,

        -- Temporal
        t.review_age_days, t.review_relative_rank, t.product_review_count,
        t.days_since_first_review, t.is_early_review, t.reviews_same_day,
        t.review_month, t.review_dayofweek, t.reviews_per_day,

        -- Lexical & language
        lx.body_word_count, lx.headline_word_count, lx.type_token_ratio,
        lx.avg_word_length, lx.exclamation_count, lx.question_count,
        lx.sentence_count_approx, lx.html_entity_density, lx.paragraph_break_count,
        lx.has_structured_body, lx.body_lang_zscore, lx.body_cat_zscore,
        lx.headline_body_ratio, lx.exclamation_density,
        lx.vine_x_marketplace, lx.verified_x_category,

        -- Embedding scalars
        em.headline_body_cosine_sim, em.title_body_cosine_sim, em.body_embedding_norm,

        -- Sentiment & quality (repetition_ratio REMOVED R10: < 0.003 importance)
        sm.body_sentiment_label, sm.body_sentiment_score, sm.body_sentiment_polarity,
        sm.headline_sentiment_label, sm.headline_sentiment_score, sm.headline_sentiment_polarity,
        sm.sentiment_mismatch, sm.flesch_reading_ease,
        sm.headline_body_jaccard, sm.title_body_jaccard,
        sm.body_bigram_diversity, sm.uppercase_ratio, sm.digit_density,
        sm.sentence_count, sm.avg_sentence_length,
        sm.flesch_lang_zscore, sm.flesch_cat_zscore, sm.polarity_lang_zscore,

        -- Evidence (price_reference_density REMOVED R10: < 0.003 importance)
        ev.measurement_count, ev.price_ref_count, ev.ordinal_comparison_count,
        ev.measurement_density, ev.ordinal_comparison_density,
        ev.quantitative_evidence_score, ev.evidence_cat_zscore,

        -- Novelty
        nv.product_tfidf_novelty, nv.product_vocab_expansion_ratio, nv.product_review_rank_novelty,

        -- Product context
        pc.product_popularity_pctile, pc.category_review_density, pc.category_product_count,
        pc.early_in_popular_product, pc.vine_in_sparse_product,
        pc.days_since_first_x_popularity, pc.verified_in_popular_product,
        pc.popularity_weight,

        -- Text specificity & length percentile
        sp.first_person_count, sp.superlative_count, sp.specific_detail_count,
        sp.first_person_density, sp.superlative_density, sp.specific_detail_density,
        sp.specificity_score, sp.body_word_count_pctile_in_product, sp.word_count_vs_product_avg,

        -- Advanced features
        -- (sentence_length_cv, product_bigram_overlap REMOVED R9: added noise)
        -- (similarity features REMOVED R10: hurt AUC after Context peak)
        -- (reviews_in_3day_window kept as burst_x_sentiment input, not a direct feature R11)
        av.vocab_overlap_with_product, av.char_ngram_entropy, av.script_homogeneity,
        av.reviews_in_3day_window, av.title_word_coverage,
        av.first_half_sentiment, av.sentiment_arc,
        av.sentiment_product_deviation, av.sentiment_lang_deviation,
        av.avg_word_length_lang_zscore, av.ttr_lang_zscore, av.sentiment_score_lang_zscore

    FROM gold.review_temporal_features   t
    LEFT JOIN gold.review_lexical_features      lx ON lx._index = t._index
    LEFT JOIN gold.review_embedding_features    em ON em._index = t._index
    LEFT JOIN gold.review_sentiment_features    sm ON sm._index = t._index
    LEFT JOIN gold.review_evidence_features     ev ON ev._index = t._index
    LEFT JOIN gold.review_novelty_features      nv ON nv._index = t._index
    LEFT JOIN gold.review_product_context       pc ON pc._index = t._index
    LEFT JOIN gold.review_specificity_features  sp ON sp._index = t._index
    LEFT JOIN gold.review_advanced_features     av ON av._index = t._index
""")

## Feature Engineering & Cleanup on ml_features

In [23]:
con.execute(f"""
    CREATE OR REPLACE TABLE gold.ml_features AS
    SELECT
        -- Identity & metadata
        _index, product_id, product_parent, label, dataset_split,
        marketplace_id, product_category_id,
        review_date, detected_language,

        -- Vine / verified binary
        CASE WHEN vine = 'Y' THEN 1.0 ELSE 0.0 END             AS vine_binary,
        CASE WHEN verified_purchase = 'Y' THEN 1.0 ELSE 0.0 END AS verified_binary,
        vine_x_marketplace,
        verified_x_category,

        -- Verified x early-review interaction
        (CASE WHEN verified_purchase = 'Y' THEN 1.0 ELSE 0.0 END)
            * CASE WHEN is_early_review THEN 1.0 ELSE 0.0 END   AS verified_early_interaction,

        -- Temporal (log-transformed)
        LN(1 + review_age_days)           AS log_review_age_days,
        review_relative_rank,
        product_review_count,
        LN(1 + days_since_first_review)   AS log_days_since_first,
        review_month,
        review_dayofweek,
        COALESCE(reviews_per_day, 0.0)    AS reviews_per_day,

        -- Lexical (log word count)
        LN(1 + body_word_count)           AS log_body_word_count,
        headline_word_count,
        CASE WHEN headline_word_count > 0 THEN 1 ELSE 0 END AS has_headline,
        type_token_ratio,
        avg_word_length,
        exclamation_density,
        sentence_count_approx,
        body_lang_zscore,
        body_cat_zscore,
        headline_body_ratio,
        LN(1 + body_word_count) * type_token_ratio AS lexical_richness,

        -- Embedding scalars
        headline_body_cosine_sim,
        title_body_cosine_sim,
        body_embedding_norm,

        -- Sentiment & quality (repetition_ratio REMOVED R10)
        body_sentiment_label,
        body_sentiment_score,
        body_sentiment_polarity,
        headline_sentiment_label,
        headline_sentiment_score,
        headline_sentiment_polarity,
        sentiment_mismatch,
        GREATEST(LEAST(flesch_reading_ease, 200.0), -200.0) AS flesch_reading_ease,
        headline_body_jaccard,
        title_body_jaccard,
        body_bigram_diversity,
        uppercase_ratio,
        digit_density,
        avg_sentence_length,
        flesch_lang_zscore,
        flesch_cat_zscore,
        polarity_lang_zscore,
        body_sentiment_polarity
            * GREATEST(LEAST(flesch_reading_ease, 200.0), -200.0) AS sentiment_readability,

        -- Evidence (price_reference_density REMOVED R10)
        evidence_cat_zscore,

        -- Novelty
        product_tfidf_novelty,
        COALESCE(product_vocab_expansion_ratio, 0.0) AS product_vocab_expansion_ratio,

        -- Product context
        product_popularity_pctile,
        category_review_density,
        category_product_count,
        days_since_first_x_popularity,
        popularity_weight,
        is_early_review,
        CASE WHEN is_early_review
             THEN COALESCE(product_popularity_pctile, 0.0)
             ELSE 0.0
        END AS early_x_popularity,

        -- Advanced features
        -- (similarity features REMOVED R10: avg/max/min/std all hurt AUC)
        -- (sentence_length_cv, product_bigram_overlap REMOVED R9: added noise)
        COALESCE(vocab_overlap_with_product, 0.5)  AS vocab_overlap_with_product,
        char_ngram_entropy,
        CAST(reviews_in_3day_window AS DOUBLE)     AS reviews_in_3day_window,
        COALESCE(title_word_coverage, 0.0)         AS title_word_coverage,
        first_half_sentiment,
        sentiment_arc,
        COALESCE(sentiment_product_deviation, 0.0) AS sentiment_product_deviation,
        COALESCE(sentiment_lang_deviation, 0.0)    AS sentiment_lang_deviation,
        COALESCE(avg_word_length_lang_zscore, 0.0) AS avg_word_length_lang_zscore,
        COALESCE(ttr_lang_zscore, 0.0)             AS ttr_lang_zscore,
        COALESCE(sentiment_score_lang_zscore, 0.0) AS sentiment_score_lang_zscore,
        COALESCE(verified_in_popular_product, false) AS verified_in_popular_product

    FROM gold.ml_features
""")

rows, cols = con.execute("""
    SELECT COUNT(*), (SELECT COUNT(*) FROM pragma_table_info('gold.ml_features'))
    FROM gold.ml_features
""").fetchone()
print(f"ml_features rebuilt: {rows:,} rows x {cols} columns")


ml_features rebuilt: 11,999 rows x 75 columns


In [24]:

total_rows, total_cols = con.execute("""
    SELECT COUNT(*), (SELECT COUNT(*) FROM pragma_table_info('gold.ml_features'))
    FROM gold.ml_features
""").fetchone()
print(f"Shape: {total_rows:,} rows x {total_cols} columns")


orphans = con.execute("""
    SELECT COUNT(*) FROM gold.ml_features WHERE _index IS NULL
""").fetchone()[0]
print(f"Orphan rows (NULL _index): {orphans}")


con.execute("""
    SELECT label, COUNT(*) AS n,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM gold.ml_features
    GROUP BY label ORDER BY label DESC
""").df()

Shape: 11,999 rows x 75 columns
Orphan rows (NULL _index): 0


,label,n,pct
0,True,4474,37.3
1,False,5139,42.8
2,<NA>,2386,19.9


In [25]:
cols = [r[0] for r in con.execute("SELECT name FROM pragma_table_info('gold.ml_features')").fetchall()]
null_exprs = ", ".join([f"ROUND(SUM(CASE WHEN \"{c}\" IS NULL THEN 1 ELSE 0 END)*100.0/COUNT(*),2) AS \"{c}\"" for c in cols])

null_rates = con.execute(f"SELECT {null_exprs} FROM gold.ml_features").df()
null_rates = null_rates.T.reset_index()
null_rates.columns = ["column_name", "null_pct"]
null_rates = null_rates[null_rates["null_pct"] > 0].sort_values("null_pct", ascending=False)
print(f"Columns with nulls: {len(null_rates)}")
null_rates


Columns with nulls: 27


,column_name,null_pct
54,product_tfidf_novelty,26.67
3,label,19.88
32,headline_body_cosine_sim,7.60
39,headline_sentiment_score,7.60
40,headline_sentiment_polarity,7.60
41,sentiment_mismatch,7.60
43,headline_body_jaccard,7.60
38,headline_sentiment_label,7.60
53,evidence_cat_zscore,2.58
18,review_month,2.25


## Export all to Parquet

In [26]:
gold_dir = Path('../../data/gold')
gold_dir.mkdir(parents=True, exist_ok=True)
today = date.today().isoformat()

exports = {
    'temporal_features':      'gold.review_temporal_features',
    'lexical_features':       'gold.review_lexical_features',
    'embedding_features':     'gold.review_embedding_features',
    'embedding_pca':          'gold.review_embedding_pca',
    'sentiment_features':     'gold.review_sentiment_features',
    'evidence_features':      'gold.review_evidence_features',
    'novelty_features':       'gold.review_novelty_features',
    'product_context':        'gold.review_product_context',
    'specificity_features':   'gold.review_specificity_features',
    'advanced_features':      'gold.review_advanced_features',
    'ml_features':            'gold.ml_features',
}

for name, table in exports.items():
    for old in gold_dir.glob(f'{name}_load_date=*.parquet'):
        old.unlink()
    out = gold_dir / f'{name}_load_date={today}.parquet'
    con.execute(f"COPY (SELECT * FROM {table}) TO '{out.as_posix()}' (FORMAT PARQUET)")
    rows = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'Saved {table} -> {out.name}  ({rows:,} rows)')

# Embeddings (raw vectors)
for old in gold_dir.glob('embeddings_load_date=*.parquet'):
    old.unlink()
emb_out = gold_dir / f'embeddings_load_date={today}.parquet'
emb_df = pd.DataFrame({
    '_index':             silver_df['_index'].values,
    'body_embedding':     list(body_emb),
    'headline_embedding': list(headline_emb),
    'title_embedding':    list(title_emb),
})
emb_df.to_parquet(emb_out, index=False)
print(f'{emb_out.name}  ({len(emb_df):,} rows x {body_emb.shape[1]} dims)')

Saved gold.review_temporal_features -> temporal_features_load_date=2026-03-15.parquet  (11,999 rows)
Saved gold.review_lexical_features -> lexical_features_load_date=2026-03-15.parquet  (11,999 rows)
Saved gold.review_embedding_features -> embedding_features_load_date=2026-03-15.parquet  (11,999 rows)
Saved gold.review_embedding_pca -> embedding_pca_load_date=2026-03-15.parquet  (11,999 rows)
Saved gold.review_sentiment_features -> sentiment_features_load_date=2026-03-15.parquet  (11,999 rows)
Saved gold.review_evidence_features -> evidence_features_load_date=2026-03-15.parquet  (11,999 rows)
Saved gold.review_novelty_features -> novelty_features_load_date=2026-03-15.parquet  (11,999 rows)
Saved gold.review_product_context -> product_context_load_date=2026-03-15.parquet  (11,999 rows)
Saved gold.review_specificity_features -> specificity_features_load_date=2026-03-15.parquet  (11,999 rows)
Saved gold.review_advanced_features -> advanced_features_load_date=2026-03-15.parquet  (11,999 ro

## Tables Created

In [27]:

tables = [
    'gold.review_temporal_features',
    'gold.review_lexical_features',
    'gold.review_embedding_features',
    'gold.review_sentiment_features',
    'gold.review_evidence_features',
    'gold.review_novelty_features',
    'gold.review_product_context',
    'gold.review_specificity_features',
    'gold.ml_features',
]

print(f'{"Table":<45} {"Rows":>8} {"Columns":>9}')
print('-' * 65)
for t in tables:
    rows = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    cols = len(con.execute(f'SELECT * FROM {t} LIMIT 0').description)
    marker = '  РІвЂ С’ ML-ready' if t == 'gold.ml_features' else ''
    print(f'{t:<45} {rows:>8,} {cols:>9}{marker}')

Table                                             Rows   Columns
-----------------------------------------------------------------
gold.review_temporal_features                   11,999        31
gold.review_lexical_features                    11,999        31
gold.review_embedding_features                  11,999         6
gold.review_sentiment_features                  11,999        26
gold.review_evidence_features                   11,999        13
gold.review_novelty_features                    11,999         7
gold.review_product_context                     11,999        14
gold.review_specificity_features                11,999        13
gold.ml_features                                11,999        75  РІвЂ С’ ML-ready


In [28]:
con.close()